In [39]:
import numpy as np
import copy
import random
import math
import time
import os
import argparse
from gurobipy import GRB, Model, quicksum

# Genetic Algorithm

In [40]:
data_inputs = [
    (40, 3),  # Index 0
    (30, 2),  # Index 1
    (100, 7), # Index 2
    (50, 4),  # Index 3
    (50, 5),  # Index 4
    (33, 3),  # Index 5
    (26, 2)   # Index 6
]
knapsack_capacity = 10  
num_items = len(data_inputs) 
# optimal_value = 140 # Target optimal value (Items 0 and 2) 
# Convert data to NumPy arrays for vectorized calculation
item_values = np.array([item[0] for item in data_inputs]) 
item_weights = np.array([item[1] for item in data_inputs])

In [41]:
pop_size = 10          # Population Size 
generations = 100      # Number of generations
cx_prob = 0.6          # Crossover probability 
mut_prob = 0.1         # Mutation probability 
elite_size = 4         # Number of elites 

In [42]:
def evaluate_fitness(individual):
    # implementation missing
    arr_np = np.array(individual)
    total_value = np.sum(item_values *arr_np)
    total_weights = np.sum(item_weights * arr_np)

    if total_weights > knapsack_capacity:
        return 0, total_weights #means bad solution
    
    return total_value, total_weights
    #pass


In [43]:
def selection(population, k=3):
    # implementation missing
    tournament = random.sample(population, k)
    selected_parent =  max(tournament)
    return selected_parent

    #pass

In [44]:
def crossover(parent1, parent2):
    # implementation missing
    crossover_point = random.randint(1, num_items - 1)

    child1 = parent1[:crossover_point] + parent2[crossover_point:]
    child2 = parent2[:crossover_point] + parent1[crossover_point:]

    return child1, child2

    #pass

In [45]:
def mutation(individual, mut_prob):
    for i in range(num_items):
        if random.random() < mut_prob:
            individual[i] = 1 - individual[i]
            #individual[i] = !individual[i]
    return individual
    # implementation missing
    pass

In [46]:
def run_ga():
    # implementation missing
    population = [[random.randint(0,1) for _ in range(num_items)]for _ in range(pop_size)]
    best_solution_history = None 
    best_fitness_histroy = 0
    
    for gen in range(generations):
        evaluated_pop = [(evaluate_fitness(ind),ind)for ind in population]
        evaluated_pop.sort(key = lambda x: x[0][0], reverse=True)
        current_best_fitness, current_best_weight = evaluated_pop[0][0]

        next_population = [list(evaluated_pop[i][1]) for i in range(elite_size)]
        
        while len(next_population) < pop_size:
            p_1, p_2 = selection(population=population, k=3), selection(population=population, k=3)
            if random .random() < cx_prob:
                c_1, c_2 = crossover(p_1,p_2)
            else:
                c_1,c_2 = list(p_1), list(p_2)
            
            next_population.append(mutation(c_1, mut_prob))
            if len(next_population) < pop_size:
                next_population.append(mutation(c_2, mut_prob))

        population = next_population
    return current_best_fitness, current_best_weight

            #pass

In [47]:
#random.seed(6)
run_ga()

(np.int64(140), np.int64(10))

# Simulated Annealing

In [48]:
values = [23, 47, 12, 35, 19, 41, 28, 16, 52, 30]
weights = [7, 11, 4, 9, 6, 10, 8, 5, 13, 9]
capacity = 30

In [49]:
def evaluate(sol):
    # implementation missing
    total_value = np.sum(values * sol)
    total_weight =  np.sum(weights * sol)
    return total_value, total_weight
    #pass

    
def neighbor(sol):
    # implementation missing
    new_sol = sol.copy()
    idx = np.random.randint(0, len(values))
    new_sol[idx] = 1-new_sol[idx]
    return new_sol
    #pass

In [50]:
# parameters
initial_temp = 1
min_temp = 0.1 # t_0/1000
iterations_per_temp = 1
cooling_rate = 0.995

current = np.zeros(len(values), dtype=int)
cur_value, cur_weight = evaluate(current)

best = current.copy()
best_value = cur_value

T = initial_temp
while T > min_temp:
    for _ in range(iterations_per_temp):
        candidate = neighbor(current)
        cand_value, cand_weight = evaluate(candidate)

        if cand_weight > capacity:
            continue

        if cand_value > cur_value:
            current = candidate.copy()
            cur_value = cand_value #so 3 values and so u can find the best solution

        else:
            delta = cand_value - cur_value
            if np.random.rand() < np.exp(delta/T):
                current = candidate.copy()
                cur_value = cand_value
        
        if cur_value >  best_value:
            best = current.copy()
            best_value = cur_value

    T *= cooling_rate
np.random.seed(1)
print(best, best_value)

# implementation missing

[1 0 0 0 0 1 0 0 1 0] 116


# Large Neighborhood Search 

In [51]:
def read_solomon_instance(file_path):

    with open(file_path, 'r') as f:
        lines = f.readlines()

    vehicle_quantity = 0
    vehicle_capacity = 0.0
    data_rows = []
    section = None
    
    # Read the instance line by line. Some other sources have XML format.
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue

        if parts[0] == 'VEHICLE':
            section = 'VEHICLE_HEADER'
            continue
        elif parts[0] == 'CUSTOMER':
            section = 'CUSTOMER_HEADER'
            continue
        
        #  For vehicles
        if section == 'VEHICLE_HEADER':
            if parts[0] == 'NUMBER':
                section = 'VEHICLE_DATA'
            continue
        
        if section == 'VEHICLE_DATA':
            vehicle_quantity = int(parts[0])
            vehicle_capacity = float(parts[1])
            section = None
            continue

        # For customers
        if section == 'CUSTOMER_HEADER':
            if parts[0] == 'CUST': 
                section = 'CUSTOMER_DATA'
            continue

        if section == 'CUSTOMER_DATA':
            if not parts[0].isdigit():
                continue
            
            row = [
                float(parts[1]),
                float(parts[2]),
                float(parts[3]),
                float(parts[4]),
                float(parts[5]),
                float(parts[6])
            ]
            data_rows.append(row)

    # Double the depot
    if data_rows:
        depot_copy = data_rows[0][:] 
        data_rows.append(depot_copy)

    data_matrix = np.array(data_rows)

    return {
        "coordinate": data_matrix[:, 0:2],       
        "demand": data_matrix[:, 2],             
        "time_window": data_matrix[:, 3:5],      
        "service_duration": data_matrix[:, 5],
        "vehicle_quantity": vehicle_quantity,
        "vehicle_capacity": vehicle_capacity
    }

In [52]:
class Customer:
    def __init__(self, id, x, y, demand, ready_time, due_date, service_time):
        self.id = id
        self.x = x
        self.y = y
        self.demand = demand
        self.ready_time = ready_time
        self.due_date = due_date
        self.service_time = service_time

class Route:
    def __init__(self, depot, capacity):
        self.route = [depot, depot] 
        self.capacity = capacity
        self.load = 0
        self.cost = 0.0
        self.is_valid = True

    def calculate_metrics(self, dist_matrix, time_matrix):
        self.load = 0
        self.cost = 0.0
        current_t = self.route[0].ready_time
        self.is_valid = True
        
        for i in range(len(self.route) - 1):
            u = self.route[i]
            v = self.route[i+1]
            
            self.cost += dist_matrix[u.id][v.id]
            
            # Time Logic
            travel = time_matrix[u.id][v.id]
            arrival_at_v = current_t + travel
            start_service_v = max(arrival_at_v, v.ready_time)
            
            if start_service_v > v.due_date:
                self.is_valid = False
                return
            
            current_t = start_service_v + v.service_time
            
            # Load Logic (Add demand of v unless it's the end depot)
            # v is the end depot if it is the last element in the list
            if i + 1 < len(self.route) - 1:
                self.load += v.demand
            
        if self.load > self.capacity:
            self.is_valid = False

class Solution:
    def __init__(self, routes, unassigned):
        self.routes = routes
        self.unassigned = unassigned
        self.total_cost = sum(r.cost for r in routes)


In [53]:
def get_dist(c1, c2):
    return math.hypot(c1.x - c2.x, c1.y -c2.y)
    # implementation missing
    #pass

In [59]:
def check_insertion(route, customer, index, dist_matrix, time_matrix, capacity):
    # implementation missing
    if route.load + customer.demand > capacity:
        return False, float("inf")
    path = route.route[:index] + [customer] + route.route[index:]

    current_time = path[0].ready_time
    u, v = route.route[index - 1], route.route[index]

    cost_change = dist_matrix[u.id][customer.id] + dist_matrix[customer.id][v.id] - dist_matrix[u.id][v.id]
    
    for i in range(len(path) - 1):
        u_node, v_node =  path[i], path[i]
        travel = time_matrix[u_node.id][v_node.id]
        arrival = current_time + travel
        start = max(arrival, v_node.ready_time)

        if start > v_node.due_date:
            return False, float("inf")
        current_time = start + v_node.service_time
    return True, cost_change

    #pass

In [60]:
def initial_solution(customers, capacity, dist_matrix, time_matrix):
    # implementation missing
    depot = customers[0]
    unassigned = set(c.id for c in customers[1:])
    routes = []
    
    while unassigned:
        curr_route = Route(depot, capacity)
        routes.append(curr_route)
        
        improved = True

        while improved:
            improved = False
            best_c, best_cost = None, float("inf")
            insert_idx = len(curr_route.route) -1
            for cid in unassigned:
                cust = customers[cid]
                possible, cost = check_insertion(curr_route,cust,insert_idx,dist_matrix,time_matrix,capacity)
                if possible and cost < best_cost:
                    best_cost = cost
                    best_c = cid
            if best_c is not None:
                curr_route.route.insert(insert_idx,customers[best_c])
                curr_route.load += customers[best_c].demand
                curr_route.cost += best_cost
                unassigned.remove(best_c)
                improved = True
    
    return Solution(routes, list(unassigned))

    #pass

In [61]:
def solve_vrptw_lns(data, max_iterations=2000, time_limit=30):
    start_time = time.time()
    
    # Adapter
    # LNS treats source and sink as a same node
    raw_coords = data["coordinate"]
    raw_demands = data["demand"]
    raw_windows = data["time_window"]
    raw_service = data["service_duration"]
    capacity = data["vehicle_capacity"]
    
    customers = []
    for i in range(len(raw_coords) - 1):
        c = Customer(i, raw_coords[i][0], raw_coords[i][1], raw_demands[i], 
                     raw_windows[i][0], raw_windows[i][1], raw_service[i])
        customers.append(c)
        
    n = len(customers)
    
    # Precompute the matrices of distances and times
    dist_matrix = np.zeros((n, n))
    time_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            d = get_dist(customers[i], customers[j])
            dist_matrix[i][j] = d
            time_matrix[i][j] = d 

    # Create initial solution
    current_sol = initial_solution(customers, capacity, dist_matrix, time_matrix)
    
    total_cost = 0
    for r in current_sol.routes:
        r.calculate_metrics(dist_matrix, time_matrix)
        total_cost += r.cost
    current_sol.total_cost = total_cost
    
    best_sol = copy.deepcopy(current_sol)
    
    # The main loop
    temp = 100.0
    cooling = 0.9995
    
    for it in range(max_iterations):
        if time.time() - start_time > time_limit:
            break
            
        new_sol = copy.deepcopy(current_sol)
        
        # Remove randomly
        n_remove = random.randint(1, min(25, n//2))
        locations = []
        for r_idx, r in enumerate(new_sol.routes):
            # Only remove customers, not the depot
            for n_idx in range(1, len(r.route) - 1):
                locations.append((r_idx, n_idx))
                
        if len(locations) > 0:
            to_remove = random.sample(locations, min(n_remove, len(locations)))
            # Remove from back to front 
            to_remove.sort(key=lambda x: (x[0], x[1]), reverse=True)
            
            removed_ids = []
            for r_i, n_i in to_remove:
                rm_node = new_sol.routes[r_i].route.pop(n_i)
                removed_ids.append(rm_node.id)
                
            # Repair
            random.shuffle(removed_ids)
            for cid in removed_ids:
                cust = customers[cid]
                b_cost, b_r, b_pos = float('inf'), -1, -1
                
                for r_idx, r in enumerate(new_sol.routes):
                    for pos in range(1, len(r.route)):
                        poss, cost = check_insertion(r, cust, pos, dist_matrix, time_matrix, capacity)
                        if poss and cost < b_cost:
                            b_cost, b_r, b_pos = cost, r_idx, pos
                
                if b_r != -1:
                    new_sol.routes[b_r].route.insert(b_pos, cust)
                else:
                    # New route
                    nr = Route(customers[0], capacity)
                    nr.route.insert(1, cust)
                    new_sol.routes.append(nr)
        
        # Recalculate and remove empty routes
        clean_routes = []
        t_cost = 0
        for r in new_sol.routes:
            r.calculate_metrics(dist_matrix, time_matrix)
            if len(r.route) > 2: # Keep non-empty routes
                clean_routes.append(r)
                t_cost += r.cost
        new_sol.routes = clean_routes
        new_sol.total_cost = t_cost
        
        # Acceptance check
        delta = new_sol.total_cost - current_sol.total_cost
        if delta < 0 or random.random() < math.exp(-delta / temp):
            current_sol = new_sol
            if current_sol.total_cost < best_sol.total_cost:
                best_sol = copy.deepcopy(current_sol)
        
        temp *= cooling

    # Format Output
    simple_routes = []
    for r in best_sol.routes:
        simple_routes.append([c.id for c in r.route])

    return {
        "status": "Feasible",
        "cost": best_sol.total_cost,
        "runtime": time.time() - start_time,
        "routes": simple_routes
    }

In [62]:
data = read_solomon_instance("c101.txt")
solve_vrptw_lns(data, time_limit=120)

{'status': 'Feasible',
 'cost': np.float64(745.6608990282205),
 'runtime': 12.841121912002563,
 'routes': [[0, 67, 24, 25, 27, 29, 11, 97, 100, 99, 0],
  [0, 98, 96, 95, 94, 10, 38, 45, 48, 51, 50, 52, 49, 47, 0],
  [0, 57, 55, 54, 62, 44, 72, 61, 64, 2, 1, 75, 0],
  [0, 32, 17, 18, 35, 74, 46, 28, 26, 23, 69, 0],
  [0, 20, 33, 41, 40, 37, 30, 39, 36, 34, 22, 21, 0],
  [0, 90, 87, 86, 83, 82, 84, 85, 88, 89, 91, 0],
  [0, 5, 3, 7, 8, 92, 93, 9, 6, 4, 0],
  [0, 81, 78, 76, 71, 70, 73, 77, 79, 80, 0],
  [0, 13, 65, 31, 19, 15, 16, 14, 12, 0],
  [0, 43, 42, 63, 53, 56, 58, 60, 59, 68, 66, 0]]}